# Actualizacion: prepare_splits v2 (Stratified Group Split)

**Cambios frente a v1:**
- Usa `StratifiedGroupKFold` de sklearn.
- Garantiza proporciones similares de positivos entre train/val/test.
- Mantiene pacientes disjuntos entre splits (igual que v1).
- Agrega validacion automatica de estratificacion.

**Esperado:** los tres splits deben tener proporciones de positivos cercanas al global (~33%),
no como el v1 donde test quedo en 20%.

**Tiempo:** < 30 segundos.

**Runtime:** CPU (no necesita GPU).


## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'
print(f'CODE_DIR: {CODE_DIR}')


## 2. Sobrescribir `scripts/prepare_splits.py` con la v2

In [ ]:
from pathlib import Path

script_content = r'''"""
scripts/prepare_splits.py  (v2)

Genera los splits train/val/test para el baseline binario del Sprint 1.

Cambio clave frente a v1: uso de StratifiedGroupKFold de sklearn para
garantizar que cada split tenga una proporcion similar de positivos/negativos
(stratified) mientras mantiene a todos los pacientes disjuntos (grouped).

Estrategia completa:
  1. Leer Data_Entry_subset_local.csv (imagenes que tenemos en disco).
  2. Filtrar por vistas aceptadas (PA/AP).
  3. Construir etiqueta binaria.
  4. Balancear con undersampling de negativos (ratio 2:1 por defecto).
  5. Asignar un label representativo por paciente (positivo si tiene al menos
     una imagen positiva, negativo en caso contrario).
  6. Split 70/15/15 con StratifiedGroupKFold.
  7. Validar: sin leakage + proporciones similares entre splits.
  8. Guardar train.csv / val.csv / test.csv.

Uso (desde Colab):
    !python /content/drive/MyDrive/Tesis_CXR/code/scripts/prepare_splits.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Permitir imports desde src/ cuando se ejecuta como script
CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.training.utils import set_seed, get_logger, load_config  # noqa: E402


# ---------------------------------------------------------------------------
# Construccion de labels
# ---------------------------------------------------------------------------
def build_binary_labels(df: pd.DataFrame, positive_class: str,
                        negative_class: str) -> pd.DataFrame:
    """
    Columna 'label':
      1 si 'positive_class' aparece en 'Finding Labels' (sola o combinada).
      0 si 'Finding Labels' == 'negative_class' exactamente.
      NaN si no cae en ninguno -> se descarta.
    """
    df = df.copy()
    is_positive = df['Finding Labels'].str.contains(positive_class, regex=False)
    is_negative = df['Finding Labels'] == negative_class

    df['label'] = np.where(is_positive, 1,
                   np.where(is_negative, 0, np.nan))
    return df


def filter_views(df: pd.DataFrame, views: list[str]) -> pd.DataFrame:
    """Deja solo las vistas aceptadas (PA, AP)."""
    return df[df['View Position'].isin(views)].copy()


def undersample_negatives(df: pd.DataFrame, ratio: float,
                          seed: int) -> pd.DataFrame:
    """
    Submuestrea la clase mayoritaria (negativa) para alcanzar:
      ratio = n_negativos / n_positivos.
    """
    positives = df[df['label'] == 1]
    negatives = df[df['label'] == 0]

    n_pos = len(positives)
    n_neg_target = min(len(negatives), int(n_pos * ratio))

    negatives_sampled = negatives.sample(n=n_neg_target, random_state=seed)
    balanced = pd.concat([positives, negatives_sampled], ignore_index=True)
    return balanced.sample(frac=1.0, random_state=seed).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Stratified Group Split (NUEVO en v2)
# ---------------------------------------------------------------------------
def assign_patient_level_label(df: pd.DataFrame) -> pd.DataFrame:
    """
    Asigna a cada paciente un 'patient_label':
      1 si tiene al menos una imagen positiva.
      0 solo si todas sus imagenes son negativas.

    Justificacion clinica: si un paciente tuvo cardiomegalia alguna vez,
    el paciente "tiene la condicion" aunque no se vea en todas sus radiografias.
    Esto tambien evita que el modelo vea al mismo paciente con etiquetas
    contradictorias en distintos splits.
    """
    patient_labels = df.groupby('Patient ID')['label'].max().astype(int)
    return patient_labels


def stratified_group_split(df: pd.DataFrame, train_ratio: float,
                           val_ratio: float, test_ratio: float,
                           seed: int, logger
                           ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split train/val/test que es simultaneamente:
      - Agrupado por paciente (no leakage entre splits).
      - Estratificado por label (proporciones similares).

    Usa StratifiedGroupKFold de sklearn en dos pasos:
      1. Primer fold: separa test del resto.
      2. Segundo fold sobre el resto: separa val de train.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Los ratios deben sumar 1.0"

    # Trabajamos a nivel imagen, sklearn se encarga de respetar los grupos
    X = df.index.to_numpy()
    y = df['label'].astype(int).to_numpy()
    groups = df['Patient ID'].to_numpy()

    # --- Paso 1: separar test ---
    # n_splits = round(1 / test_ratio). Con test_ratio=0.15 -> 7 folds
    # (cada fold es ~14% del total). Tomamos el primer fold como test.
    n_splits_test = max(2, round(1.0 / test_ratio))
    sgkf_test = StratifiedGroupKFold(
        n_splits=n_splits_test, shuffle=True, random_state=seed,
    )
    train_val_idx, test_idx = next(sgkf_test.split(X, y, groups))

    df_test = df.iloc[test_idx].reset_index(drop=True)
    df_train_val = df.iloc[train_val_idx].reset_index(drop=True)

    # --- Paso 2: separar val de train_val ---
    # val_ratio sobre el total = val_ratio_rel sobre train_val
    # val_ratio_rel = val_ratio / (train_ratio + val_ratio)
    val_ratio_rel = val_ratio / (train_ratio + val_ratio)
    n_splits_val = max(2, round(1.0 / val_ratio_rel))

    X_tv = df_train_val.index.to_numpy()
    y_tv = df_train_val['label'].astype(int).to_numpy()
    groups_tv = df_train_val['Patient ID'].to_numpy()

    sgkf_val = StratifiedGroupKFold(
        n_splits=n_splits_val, shuffle=True, random_state=seed,
    )
    train_idx, val_idx = next(sgkf_val.split(X_tv, y_tv, groups_tv))

    df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
    df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

    logger.info(
        "Split efectuado con StratifiedGroupKFold "
        "(n_splits_test=%d, n_splits_val=%d)",
        n_splits_test, n_splits_val,
    )

    return df_train, df_val, df_test


# ---------------------------------------------------------------------------
# Reportes
# ---------------------------------------------------------------------------
def report_split(name: str, df: pd.DataFrame, logger) -> None:
    n = len(df)
    n_pos = int((df['label'] == 1).sum())
    n_neg = int((df['label'] == 0).sum())
    n_patients = df['Patient ID'].nunique()
    pct_pos = (n_pos / n * 100) if n else 0.0

    logger.info(
        "%-5s: %4d imgs | %3d pacientes | pos=%3d (%.1f%%) | neg=%3d",
        name.upper(), n, n_patients, n_pos, pct_pos, n_neg,
    )


def check_no_leakage(train: pd.DataFrame, val: pd.DataFrame,
                     test: pd.DataFrame, logger) -> None:
    p_train = set(train['Patient ID'])
    p_val = set(val['Patient ID'])
    p_test = set(test['Patient ID'])

    overlaps = {
        'train-val': p_train & p_val,
        'train-test': p_train & p_test,
        'val-test': p_val & p_test,
    }
    for name, overlap in overlaps.items():
        if overlap:
            logger.error("LEAKAGE en %s: %d pacientes compartidos", name, len(overlap))
            raise AssertionError(f"Data leakage detectado en {name}")

    logger.info("Sin leakage: pacientes disjuntos entre splits.")


def check_stratification(train: pd.DataFrame, val: pd.DataFrame,
                         test: pd.DataFrame, logger,
                         max_deviation: float = 10.0) -> None:
    """
    Alerta si alguna proporcion de positivos se desvia mas de max_deviation
    puntos porcentuales respecto al global. No es un error fatal, solo aviso.
    """
    combined = pd.concat([train, val, test], ignore_index=True)
    global_pct = (combined['label'] == 1).mean() * 100

    logger.info("Proporcion global de positivos: %.1f%%", global_pct)

    for name, df in [("train", train), ("val", val), ("test", test)]:
        pct = (df['label'] == 1).mean() * 100
        deviation = abs(pct - global_pct)
        if deviation > max_deviation:
            logger.warning(
                "%s tiene %.1f%% positivos (desviacion %.1fpp del global). "
                "Posible desbalance.",
                name, pct, deviation,
            )


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main(config_path: str | None = None) -> None:
    if config_path is None:
        config_path = CODE_DIR / "configs" / "baseline.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project['seed'])
    log_file = Path(cfg.paths['logs']) / "prepare_splits.log"
    logger = get_logger("prepare_splits", log_file=log_file)

    logger.info("=" * 60)
    logger.info("PREPARE SPLITS v2 - Stratified Group Split")
    logger.info("=" * 60)

    nih_dir = Path(cfg.paths['data_raw'])
    processed_dir = Path(cfg.paths['data_processed'])
    processed_dir.mkdir(parents=True, exist_ok=True)

    subset_csv = nih_dir / "Data_Entry_subset_local.csv"
    if not subset_csv.exists():
        raise FileNotFoundError(
            f"No se encontro {subset_csv}. "
            "Corre primero 00b_download_nih.ipynb."
        )

    # 1. Cargar
    df = pd.read_csv(subset_csv)
    logger.info("CSV cargado: %d filas", len(df))

    # 2. Filtrar vistas
    df = filter_views(df, cfg.dataset['views'])
    logger.info("Tras filtrar vistas %s: %d filas", cfg.dataset['views'], len(df))

    # 3. Construir labels
    df = build_binary_labels(
        df,
        positive_class=cfg.dataset['positive_class'],
        negative_class=cfg.dataset['negative_class'],
    )
    n_before = len(df)
    df = df.dropna(subset=['label']).copy()
    df['label'] = df['label'].astype(int)
    logger.info(
        "Tras construir labels: %d filas (descartadas %d ambiguas)",
        len(df), n_before - len(df),
    )
    logger.info(
        "  Positivos (%s): %d | Negativos (%s): %d",
        cfg.dataset['positive_class'], int((df['label'] == 1).sum()),
        cfg.dataset['negative_class'], int((df['label'] == 0).sum()),
    )

    # 4. Balancear
    ratio = cfg.get('class_ratio', 2.0)
    df_balanced = undersample_negatives(df, ratio=ratio, seed=cfg.project['seed'])
    logger.info("Tras balancear (ratio %.1f:1): %d filas", ratio, len(df_balanced))

    # 5. Info a nivel paciente (para debug)
    patient_labels = assign_patient_level_label(df_balanced)
    n_pat_pos = int((patient_labels == 1).sum())
    n_pat_neg = int((patient_labels == 0).sum())
    logger.info(
        "Pacientes unicos: %d | con positivo: %d | solo negativos: %d",
        len(patient_labels), n_pat_pos, n_pat_neg,
    )

    # 6. Split estratificado por grupo
    train_df, val_df, test_df = stratified_group_split(
        df_balanced,
        train_ratio=cfg.splits['train_ratio'],
        val_ratio=cfg.splits['val_ratio'],
        test_ratio=cfg.splits['test_ratio'],
        seed=cfg.project['seed'],
        logger=logger,
    )

    # 7. Reportar y validar
    logger.info("-" * 60)
    report_split("train", train_df, logger)
    report_split("val", val_df, logger)
    report_split("test", test_df, logger)
    logger.info("-" * 60)
    check_no_leakage(train_df, val_df, test_df, logger)
    check_stratification(train_df, val_df, test_df, logger)

    # 8. Guardar
    columns_to_save = ['Image Index', 'Finding Labels', 'Patient ID',
                       'View Position', 'label']
    train_df[columns_to_save].to_csv(processed_dir / "train.csv", index=False)
    val_df[columns_to_save].to_csv(processed_dir / "val.csv", index=False)
    test_df[columns_to_save].to_csv(processed_dir / "test.csv", index=False)

    logger.info("Splits guardados en: %s", processed_dir)
    logger.info("  train.csv: %d filas", len(train_df))
    logger.info("  val.csv:   %d filas", len(val_df))
    logger.info("  test.csv:  %d filas", len(test_df))
    logger.info("=" * 60)
    logger.info("PREPARE SPLITS v2 OK")


if __name__ == "__main__":
    main()
'''

script_path = f'{CODE_DIR}/scripts/prepare_splits.py'
Path(script_path).write_text(script_content, encoding='utf-8')
print(f'OK -> {script_path} ({len(script_content):,} chars)')


## 3. Ejecutar el script v2

Debe correr sin errores. Presta atencion al reporte final: las proporciones de positivos
de los tres splits deben ser similares entre si (diferencia < 10 puntos porcentuales).

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/prepare_splits.py'],
    capture_output=True, text=True
)
print('STDOUT:')
print(result.stdout)
print()
print('STDERR (donde va el log):')
print(result.stderr)


## 4. Verificar que los CSVs se sobrescribieron bien

In [ ]:
import pandas as pd

processed = f'{PROJECT_ROOT}/data/processed'

print('RESUMEN DE LOS NUEVOS SPLITS:')
print('-' * 60)
for name in ['train', 'val', 'test']:
    df = pd.read_csv(f'{processed}/{name}.csv')
    n_pos = (df['label'] == 1).sum()
    n_neg = (df['label'] == 0).sum()
    n_pat = df['Patient ID'].nunique()
    pct = n_pos / len(df) * 100
    print(f'{name:5s}: {len(df):4d} imgs | {n_pat:3d} pacientes | '
          f'pos={n_pos:3d} ({pct:.1f}%) | neg={n_neg:3d}')

print()
print('COMPARACION ESPERADA:')
print('  Las tres proporciones de positivos deberian estar entre ~28% y ~40%.')
print('  En v1 teniamos: train=34.3%, val=41.7%, test=19.8% (test muy bajo).')
print('  En v2 esperamos: los tres cerca de 33%.')


## 5. Siguiente paso

Si las proporciones quedaron balanceadas (todas cerca del 33%),
vuelve al chat para recibir:

1. `src/datasets/transforms.py` - preprocesamiento de imagenes.
2. `src/datasets/nih.py` - Dataset de PyTorch que lee los CSVs y carga imagenes.
3. Notebook de EDA para visualizar los primeros ejemplos preprocesados.

Con esos tres archivos, el pipeline de datos queda completo y pasamos al modelo.
